Here We importing important liberarys to use them into ferther work

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

Importing Data

In [ ]:
index_names = ['unit_number', 'time_cycles']
setting_names = ['setting_1', 'setting_2', 'setting_3']
sensor_names = ['s_{}'.format(i+1) for i in range(0,21)]
col_names = index_names + setting_names + sensor_names

In [ ]:
dftrain = pd.read_csv('/Users/soujanyadutta/RUL_Prediction_Nasa_Turbofan_Engine/Data/train_FD001.txt',sep='\s+',header=None,index_col=False,names=col_names)
dfvalid = pd.read_csv('/Users/soujanyadutta/RUL_Prediction_Nasa_Turbofan_Engine/Data/test_FD001.txt',sep='\s+',header=None,index_col=False,names=col_names)
y_valid = pd.read_csv('/Users/soujanyadutta/RUL_Prediction_Nasa_Turbofan_Engine/Data/RUL_FD001.txt',sep='\s+',header=None,index_col=False,names=['RUL'])
dfvalid.shape

In [ ]:
train = dftrain.copy()
valid = dfvalid.copy()

In [ ]:
train

Defining Sensor Columns

In [ ]:
Sensor_dictionary={}
dict_list=[ "(Fan inlet temperature) (◦R)",
"(LPC outlet temperature) (◦R)",
"(HPC outlet temperature) (◦R)",
"(LPT outlet temperature) (◦R)",
"(Fan inlet Pressure) (psia)",
"(bypass-duct pressure) (psia)",
"(HPC outlet pressure) (psia)",
"(Physical fan speed) (rpm)",
"(Physical core speed) (rpm)",
"(Engine pressure ratio(P50/P2)",
"(HPC outlet Static pressure) (psia)",
"(Ratio of fuel flow to Ps30) (pps/psia)",
"(Corrected fan speed) (rpm)",
"(Corrected core speed) (rpm)",
"(Bypass Ratio) ",
"(Burner fuel-air ratio)",
"(Bleed Enthalpy)",
"(Required fan speed)",
"(Required fan conversion speed)",
"(High-pressure turbines Cool air flow)",
"(Low-pressure turbines Cool air flow)" ]
i=1
for x in dict_list :
    Sensor_dictionary['s_'+str(i)]=x
    i+=1
Sensor_dictionary

In [ ]:
train['time_cycles'].value_counts()

In [ ]:
#Cheking the presence of Nan values
print('Total None values in the train dataset : ',train.isna().sum())

In [ ]:
train.loc[:,['unit_number','time_cycles']].describe()

In [ ]:
train.loc[:,'s_1':].describe().transpose()

In [ ]:
# Delete the Columns where verience is '0'
train = train.drop(['s_1','s_5','s_10','s_16','s_18','s_19'], axis=1)

train

In [ ]:
train_grouped_by_unit=train.groupby('unit_number').max()
max_time_cycle=train_grouped_by_unit['time_cycles']
marged = train.merge(max_time_cycle.to_frame(name='max_time_cycle'), left_on='unit_number', right_index=True)
marged['RUL'] = marged['max_time_cycle'] - marged['time_cycles']
marged = marged.drop('max_time_cycle', axis=1)
train = marged
train

In [ ]:
train.corr()

**Correlation heatmap**

In [ ]:
corr = train.corr()

#Heatmap

sns.heatmap(corr,annot=False,cmap='coolwarm',linewidths = 0.5)

plt.title('correlation heatmap')
plt.show()

**Pairplot**

In [ ]:
sns.pairplot(train)
plt.show()

**Box Plot**

In [ ]:
for i, col in enumerate(train):
    sns.boxplot(x=train[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

**Skewness Distribution**

In [ ]:
for i, col in enumerate(train):
    sns.histplot(train[col], kde=True)
    skew_val = train[col].skew()
    plt.title(f"{col}\nSkewness: {skew_val:.3f}")
    plt.show()

# Feature Enginnering

**Feature engineering focuses on:**

1.	Cleaning and selecting useful sensors

2.	Normalizing & de-noising

3.	Generating time-series features (lags, rolling windows)

4.	Health-index transformations

5.	Aggregates for ML models (if using non-sequence models like XGBoost)


 Remove Sensors With No Trend / Low Variance
In C-MAPSS, some sensors do not relate to degradation.


In [ ]:
#low_var = train.var().sort_values()
#print(low_var)

# Or
corr = train.corr()['RUL'].sort_values()
print(corr)

From my visual cheking of that uploaded dataset I can figure it out that these rows have low corr with or can say useless towarads calculating RUL, so ignoring it.  { s1, s5, s10, s16, s18, s19 }

**Normalize Sensor Readings Per Operating Condition**

Because different engines run under different “settings”, normalization must be done within each operating condition:

In [ ]:
#use this code
from sklearn.preprocessing import StandardScaler

sensor_cols = ['s_2', 's_3', 's_4', 's_6', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21']
scalers = {}
for cond, group in train.groupby(['setting_1', 'setting_2', 'setting_3']):
    scaler = StandardScaler()
    train.loc[group.index, sensor_cols] = scaler.fit_transform(group[sensor_cols])
    scalers[cond] = scaler


examples: Because sensor 2(high alt) != sensor 2(low alt)

Now we need to design some Degredation model, that will something that  most probalby fit with that data.

 **Add “Health Index” Features**
A Health Index (HI) is a compact representation of degradation.


A. Linear degradation index


In [ ]:
train['cycle_norm'] = train.groupby('unit_number')['time_cycles'].transform(
    lambda x: x / x.max()
)

B. PCA Health Index (very effective)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=1)
train['HI_pca'] = pca.fit_transform(train[sensor_cols])


C. Exponential degradation transform

In [ ]:
import numpy as np
train['exp_cycle'] = np.exp(train['time_cycles'] / train.groupby('unit_number')['time_cycles'].transform('max'))

**Exponential degradation transform**

A turbofan's condition depends on recent sensor trends, not just the current cycle.

Typically use windows: 5, 10, 20 cycles

Example for each sensor:

windows = [5, 10, 20]


This adds:

•	rolling mean

•	rolling slope (trend)

•	rolling variability


In [ ]:
windows = [5, 10, 20]
for w in windows:
  for col in sensor_cols:
    # Rolling mean
    train[f"{col}_Rolling_mean_{w}"] = train.groupby('unit_number')[col].transform(
        lambda x: x.rolling(w,min_periods=1).mean()
    )
    # Rolling diffrence
    train[f"{col}_Rolling_diffrence_{w}"] = train.groupby('unit_number')[col].transform(
        lambda x: x.diff().rolling(w,min_periods=1).mean()
    )

Sensor Deltas (changes over time)

Degradation is captured by how fast a sensor is drifting.


In [ ]:
for col in sensor_cols:
    train[f'{col}_delta'] = train.groupby('unit_number')[col].diff()

#Also differences at longer horizons:
train[f'{col}_delta_5'] = train.groupby('unit_number')[col].diff(5)
train[f'{col}_delta_10'] = train.groupby('unit_number')[col].diff(10)
train[f'{col}_delta_20'] = train.groupby('unit_number')[col].diff(20)


**Operating Condition Encodings**

using ML models (XGBoost, Random Forest), create encodings:

In [ ]:
train['op_mode'] = (
    train['setting_1'].round(2).astype(str) + '_' +
    train['setting_2'].round(2).astype(str) + '_' +
    train['setting_3'].round(2).astype(str)
)
train = pd.get_dummies(train, columns=['op_mode'])

train.to_csv('train_processed.csv', index=False)

**Windowed Time-Series Tensor (for LSTM/GRU)**

If you're training an LSTM/Transformer, reshape into sequences:

In [ ]:
def create_sequences(train, window=30):
    sequences = []
    labels = []
    for eng_id, group in train.groupby("unit_number"):
        data = group.values
        for i in range(len(group)-window):
            sequences.append(data[i:i+window])
            labels.append(group['RUL'].iloc[i+window])
    return np.array(sequences), np.array(labels)


In [ ]:
train

Use window sizes like 20–50 cycles.